# ClaimGuard — re-run of the experiments that use the fine-tuned verifier

The first full run (23 Nov) trained the verifier in float16 because recent `transformers` versions load a
checkpoint in the dtype it was saved in: every training run had `loss=nan`. The code now forces float32 and
stops at the first NaN. The steps that did **not** use the fine-tuned model (retrieval, no-evidence baselines,
zero-shot NLI, rules, multilingual French, LLM-AggreFact zero-shot and rules, LLM judge) were correct and are
**not** re-run here: their results come from the first run.

Same setup as before: input dataset **claimguard-code** (upload the NEW `claimguard.zip` as a new version of the
dataset), GPU T4 x2, Internet on, secret HF_TOKEN. Then *Save Version → Save & Run All*. About 1 h 15.

## 0. Setup

In [ ]:
import glob, json, os, shutil, subprocess, time
from pathlib import Path

# find the project inside the uploaded dataset (Kaggle unzips it, the folder depth can vary)
candidates = [Path(p).parent for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
              if "claimguard" in Path(p).read_text()]
assert candidates, "claimguard.zip not found: add it as an input dataset (see the first cell)"
SRC = candidates[0]
WORK = Path("/kaggle/working/claimguard")
if not WORK.exists():
    shutil.copytree(SRC, WORK)   # the input folder is read-only, so work on a copy
os.chdir(WORK)
print("project copied from", SRC)

In [ ]:
!pip install -q -r requirements-dev.txt && pip install -q -e . && pip install -q bitsandbytes accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: set the accelerator to GPU T4 x2")

In [ ]:
# run a shell command, log how long it took and if it worked, and NEVER stop the notebook
STATUS_FILE = WORK / "results" / "run_status.json"
STATUS_FILE.parent.mkdir(exist_ok=True)
status = json.loads(STATUS_FILE.read_text()) if STATUS_FILE.exists() else []

def run(name, cmd):
    print(f"\n===== {name} =====\n$ {cmd}", flush=True)
    start = time.time()
    proc = subprocess.run(cmd, shell=True, cwd=WORK, text=True, capture_output=True)
    minutes = round((time.time() - start) / 60, 1)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print("STDERR:", proc.stderr[-3000:])
    status.append({"step": name, "ok": proc.returncode == 0, "minutes": minutes,
                   "error": None if proc.returncode == 0 else proc.stderr[-1500:]})
    STATUS_FILE.write_text(json.dumps(status, indent=2))
    print(f"-> {'OK' if proc.returncode == 0 else 'FAILED'} in {minutes} min", flush=True)
    return proc.returncode == 0

In [ ]:
# Hugging Face login from the Kaggle secret (only LLM-AggreFact needs it, the rest works without)
HF_OK = False
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    HF_OK = True
    print("Hugging Face login OK")
except Exception as err:
    print("no HF_TOKEN secret, LLM-AggreFact will be skipped:", type(err).__name__)

## 1. Tests and data

In [ ]:
run("unit tests", "make test")
run("data", "make data && cat data/processed/scifact/stats.json")

## 4. Hyperparameter search (12 runs), then the best setting is written in the config automatically

In [ ]:
import yaml
if run("sweep", "python scripts/train_verifier.py --config configs/train_verifier.yaml --sweep"):
    rows = json.loads(Path("results/sweep.json").read_text())
    best = max(rows, key=lambda r: r["dev_macro_f1"])
    cfg_path = Path("configs/train_verifier.yaml")
    print("best on dev:", best)
    # change only the 3 numbers of the train: block (their first occurrence), keep the comments.
    # lr is written as 1.0e-05 and not 1e-05: YAML would read "1e-05" as a string
    import re
    text = cfg_path.read_text()
    values = {"lr": f"{float(best['lr']):.1e}", "batch_size": str(int(best["batch_size"])), "epochs": str(int(best["epochs"]))}
    for key, value in values.items():
        text = re.sub(rf"^  {key}: .*$", f"  {key}: {value}", text, count=1, flags=re.MULTILINE)
    cfg_path.write_text(text)
    check = yaml.safe_load(cfg_path.read_text())["train"]
    assert float(check["lr"]) == float(best["lr"]) and check["batch_size"] == best["batch_size"] and check["epochs"] == best["epochs"]
    print("config updated:", {k: check[k] for k in ("lr", "batch_size", "epochs")})
    shutil.copy(cfg_path, "results/train_verifier_used.yaml")   # keep a trace of the config really used
shutil.rmtree("models/sweep_tmp", ignore_errors=True)            # 700 MB temporary checkpoint

## 5. Train the verifier with 3 seeds

In [ ]:
run("train 3 seeds", "python scripts/train_verifier.py --config configs/train_verifier.yaml --all-seeds")
log = json.loads(Path("models/verifier/train_log.json").read_text())
print("dev macro-F1 per epoch:", [round(h["macro_f1"], 3) for h in log["history"]])
assert log["best_dev_macro_f1"] > 0.3, "the verifier did not learn (macro-F1 <= 0.3): stop and send the log"

## 6. Calibration (all choices on dev), main pipeline with ablations, the 3 seeds compared

In [ ]:
run("calibration + dev choices", "python scripts/evaluate.py calibration --config configs/default.yaml")
run("main pipeline + ablations", "python scripts/evaluate.py pipeline --config configs/default.yaml")
for seed, folder in ((13, "models/verifier"), (42, "models/verifier-seed42"), (2026, "models/verifier-seed2026")):
    run(f"pipeline seed {seed}", f"python scripts/evaluate.py pipeline --config configs/default.yaml --verifier-model {folder} --tag seed{seed}")

## 7. Ablation A5: fine-tuning from a plain encoder (no NLI pre-training)

In [ ]:
run("train plain encoder", "python scripts/train_verifier.py --config configs/train_verifier.yaml --base-model microsoft/deberta-v3-base --out-dir models/verifier-plain")
run("pipeline plain encoder", "python scripts/evaluate.py pipeline --config configs/default.yaml --verifier-model models/verifier-plain --tag plain_encoder")

## 8. Stress tests and efficiency

In [ ]:
for ds in ("contrast", "climate", "french"):
    run(f"stress {ds}", f"python scripts/evaluate.py given --dataset {ds} --config configs/default.yaml")
run("efficiency FP32 vs INT8", "python scripts/evaluate.py efficiency --config configs/default.yaml")

## 9. Extension: sentences written by LLMs (LLM-AggreFact, gated dataset)

In [ ]:
if HF_OK:
    run("aggrefact fine-tuned", "python scripts/evaluate.py aggrefact --config configs/default.yaml --per-dataset 100")
else:
    print("skipped: add the HF_TOKEN secret and re-run only this section")

## 11. Error export, tables, figures, and what to keep (tables will be rebuilt with the results of both runs)

In [ ]:
run("errors export", "python scripts/evaluate.py errors --config configs/default.yaml")
run("tables", "python scripts/make_tables.py")
run("figures", "python scripts/make_figures.py")
# keep only the main verifier (needed for the demo); the other checkpoints already gave their results
for folder in ("models/verifier-seed42", "models/verifier-seed2026", "models/verifier-plain"):
    shutil.rmtree(folder, ignore_errors=True)
shutil.make_archive("/kaggle/working/results", "zip", WORK / "results")
shutil.make_archive("/kaggle/working/verifier", "zip", WORK / "models")
print("\n".join(f"{'OK ' if s['ok'] else 'FAIL'} {s['minutes']:>6} min  {s['step']}" for s in status))
print("\ntotal:", round(sum(s["minutes"] for s in status), 1), "min")

In [ ]:
from IPython.display import Markdown
Markdown(Path("results/tables.md").read_text() if Path("results/tables.md").exists() else "no tables")